In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path
import numpy as onp
from tqdm import tqdm

from msmjax.utils.benchmarking import (
    path_input_structures,
    calc_nonperiodic_reference_results,
    eval_lammps_pppm,
)

In [2]:
LAMMPS_EXECUTABLE = "/home/florian/Downloads/lammps-static/bin/lmp"

# Non-periodic

In [3]:
n_particles = 10000
outdir = Path("nonperiodic/")

outdir.mkdir(parents=True, exist_ok=True)
structures = onp.load(path_input_structures / f"structures_{n_particles}.npz")
n_structures = structures["positions"].shape[0]
all_positions = structures["positions"].astype("float64")
all_charges = structures["charges"].astype("float64")
all_cells = structures["cells"].astype("float64")

all_energies = onp.full(n_structures, onp.nan)
all_forces = onp.full((n_structures, n_particles, 3), onp.nan)
all_chargegrads = onp.full((n_structures, n_particles), onp.nan)
all_stresses = onp.full((n_structures, 6), onp.nan)
for i in tqdm(range(n_structures)):
    pos = all_positions[i]
    chg = all_charges[i]
    cll = all_cells[i]
    energy, forces, chargegrad, stress = calc_nonperiodic_reference_results(
        pos, chg, cll
    )
    all_energies[i] = energy
    all_forces[i] = forces
    all_chargegrads[i] = chargegrad
    all_stresses[i] = stress

onp.savez_compressed(
    outdir / "structures.npz",
    positions=all_positions,
    charges=all_charges,
    cells=all_cells,
)
onp.savez_compressed(
    outdir / "reference_results.npz",
    energies=all_energies,
    forces=all_forces,
    chargegrads=all_chargegrads,
    stresses=all_stresses,
)

100%|██████████| 11/11 [00:03<00:00,  3.20it/s]


# Periodic

In [4]:
n_particles = 100
outdir = Path("periodic/")

outdir.mkdir(parents=True, exist_ok=True)
structures = onp.load(path_input_structures / f"structures_{n_particles}.npz")
n_structures = structures["positions"].shape[0]
all_positions = structures["positions"].astype("float64")
all_charges = structures["charges"].astype("float64")
all_cells = structures["cells"].astype("float64")

all_energies = onp.full(n_structures, onp.nan)
all_forces = onp.full((n_structures, n_particles, 3), onp.nan)
all_chargegrads = onp.full((n_structures, n_particles), onp.nan)
all_stresses = onp.full((n_structures, 6), onp.nan)
for i in tqdm(range(n_structures)):
    pos = all_positions[i]
    chg = all_charges[i]
    cll = all_cells[i]
    energy, forces, chargegrad, stress = eval_lammps_pppm(
        pos,
        chg,
        cll,
        LAMMPS_EXECUTABLE,
        accuracy=1.0e-8,
        max_neighbors_one_atom=10000,
    )
    all_energies[i] = energy
    all_forces[i] = forces
    all_chargegrads[i] = chargegrad
    all_stresses[i] = stress

onp.savez_compressed(
    outdir / "structures.npz",
    positions=all_positions,
    charges=all_charges,
    cells=all_cells,
)
onp.savez_compressed(
    outdir / "reference_results.npz",
    energies=all_energies,
    forces=all_forces,
    chargegrads=all_chargegrads,
    stresses=all_stresses,
)

100%|██████████| 11/11 [00:00<00:00, 29.66it/s]
